In [1]:
import requests
import datetime
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor

In [2]:
# API Key 
API_KEY = 'ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8'

In [3]:
# Maximum API calls per minute
API_LIMIT = 300

In [4]:
def get_all_tickers():
    """
    Fetches a list of all investable stocks and major indexes from Financial Modeling Prep API.
    """
    stock_url = f"https://financialmodelingprep.com/api/v3/stock/list?apikey={API_KEY}"
    index_url = f"https://financialmodelingprep.com/api/v3/symbol/available-indexes?apikey={API_KEY}"

    try:
        stock_response = requests.get(stock_url).json()
        index_response = requests.get(index_url).json()

        stock_tickers = [item['symbol'] for item in stock_response]
        index_tickers = [item['symbol'] for item in index_response]

        return stock_tickers + index_tickers  # Combine stock and index tickers
    except Exception as e:
        print("Error fetching tickers:", e)
        return []

In [5]:
def get_sd(ticker):
    """
    Fetches standard deviation for a stock over 1Y, 5Y, and 10Y.
    """
    base_url = 'https://financialmodelingprep.com/api/v3/technical_indicator/1day/'
    periods = {"1Y": 252, "5Y": 1260, "10Y": 2520}
    sd_values = {}

    for key, period in periods.items():
        url = f"{base_url}{ticker}?type=standardDeviation&period={period}&apikey={API_KEY}"
        try:
            response = requests.get(url)
            data = response.json()

            if isinstance(data, list) and len(data) > 0:
                sd_values[key] = round(data[0].get('standardDeviation', None), 4)
            else:
                sd_values[key] = None
        except Exception as e:
            print(f"Error fetching SD for {ticker}: {e}")
            sd_values[key] = None

    return sd_values



In [6]:
def get_cagr(ticker):
    """
    Fetches CAGR for a stock over 1Y, 5Y, and 10Y.
    """
    base_url = 'https://financialmodelingprep.com/api/v3/historical-price-full/'
    years_list = [1, 5, 10]
    cagr_values = {}

    for years in years_list:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=years * 365)

        url = f"{base_url}{ticker}?from={start_date}&to={end_date}&apikey={API_KEY}"
        try:
            response = requests.get(url)
            data = response.json()

            if "historical" in data and len(data["historical"]) > 0:
                historical_data = sorted(data["historical"], key=lambda x: x["date"])
                P_start = historical_data[0]["close"]
                P_end = historical_data[-1]["close"]

                cagr = ((P_end / P_start) ** (1 / years)) - 1
                cagr_values[f"{years}Y"] = round(cagr * 100, 2)
            else:
                cagr_values[f"{years}Y"] = None
        except Exception as e:
            print(f"Error fetching CAGR for {ticker}: {e}")
            cagr_values[f"{years}Y"] = None

    return cagr_values


In [7]:
def get_stock_data(ticker):
    """
    Combines Standard Deviation and CAGR data into a single dictionary per stock.
    """
    try:
        sd_data = get_sd(ticker)
        cagr_data = get_cagr(ticker)

        return {
            "Ticker": ticker,
            "1Y SD": sd_data["1Y"],
            "5Y SD": sd_data["5Y"],
            "10Y SD": sd_data["10Y"],
            "1Y CAGR": cagr_data["1Y"],
            "5Y CAGR": cagr_data["5Y"],
            "10Y CAGR": cagr_data["10Y"]
        }
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        return None

In [8]:
def get_multiple_stocks_data(tickers, workers=30):
    """
    Fetches SD and CAGR for multiple stocks using multithreading.
    Uses up to `workers` threads for parallel requests.
    """
    total_tickers = len(tickers)
    all_data = []

    print(f"Processing {total_tickers} tickers using {workers} threads...")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(get_stock_data, tickers))

    # Filter out None values (failed requests)
    all_data = [result for result in results if result]

    # Convert to DataFrame
    df = pd.DataFrame(all_data)

    # Save interim results
    df.to_csv("all_stocks_sd_cagr.csv", index=False)

    return df

In [9]:
# Get
all_tickers = get_all_tickers()

# Process
df = get_multiple_stocks_data(all_tickers, workers=50)  # Using 50 threads for speed

# Save
df.to_csv("all_stocks_sd_cagr.csv", index=False)

# Display
print(df.head())

Processing 85099 tickers using 50 threads...


KeyboardInterrupt: 